# Feature Selection - All 15 Use Cases
**DNA Gene Mapping Project - ML Phase V5**  
**Author:** Sharique Mohammad  
**Date:** February 2026  

## Objective
Apply feature selection to all 15 use cases before model training.
Output one approved feature list CSV per use case to data/feature_lists/.

## Three-Pass Filter Applied to Every Use Case

Pass 1 - Leakage removal: remove ID columns, label-derived columns, and
columns that directly encode the target or its clinical equivalent.

Pass 2 - Quality filter: remove columns with more than 50% missing values
in the training split, and remove zero-variance columns.

Pass 3 - Correlation filter: remove one of each pair of features with
Pearson r above 0.95, keeping the more interpretable feature.

## Output Per Use Case
data/feature_lists/
  uc01_clinical_features.csv
  uc02_disease_features.csv
  uc03_pharmacogene_features.csv
  uc04_variant_impact_features.csv
  uc05_structural_variant_features.csv
  uc06_drug_response_variant_features.csv
  uc07_cancer_variant_features.csv
  uc08_carrier_screening_features.csv
  uc09_population_frequency_features.csv
  uc10_gene_pharmacogene_features.csv
  uc11_gene_expression_features.csv
  uc12_protein_family_features.csv
  uc13_gene_test_features.csv
  uc14_expression_simple_features.csv
  uc15_cancer_molecular_features.csv
  feature_selection_summary.csv

## Use Cases and Source Tables
UC01 - clinical_ml_features            - target: target_is_pathogenic
UC02 - disease_ml_features             - target: is_pathogenic
UC03 - pharmacogene_ml_features        - target: is_pathogenic
UC04 - variant_impact_ml_features      - target: is_high_impact
UC05 - structural_variant_ml_features  - target: sv_classification (multiclass)
UC06 - variant_drug_response_ml_features - target: is_actionable_pharmacogene_variant
UC07 - variant_cancer_ml_features      - target: is_driver_candidate
UC08 - variant_population_ml_features  - target: is_carrier_screening_candidate
UC09 - population_frequency_ml_features - target: is_clinically_actionable_rare_variant
UC10 - gene_pharmacogene_ml_features   - target: is_high_priority_pharmacogene
UC11 - gene_expression_ml_features     - target: is_clinically_relevant_expression
UC12 - gene_protein_family_ml_features - target: is_high_value_protein_family
UC13 - gene_test_availability_ml_features - target: is_high_priority_test_gene
UC14 - transcript_expression_ml_features  - target: is_clinically_relevant_expression
UC15 - cancer_variant_ml_features      - target: gene_cancer_role (multiclass)

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os
from pathlib import Path
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

PROJECT_ROOT    = Path().absolute().parent.parent
FEATURE_DIR     = PROJECT_ROOT / 'data' / 'feature_lists'
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete")
print(f"Feature lists output: {FEATURE_DIR}")

## 2. Database Connection

In [ ]:
POSTGRES_HOST     = os.getenv("POSTGRES_HOST")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT")
POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")

conn_str = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(conn_str)

print("Database connection established")
print(f"Host : {POSTGRES_HOST}:{POSTGRES_PORT}")
print(f"DB   : {POSTGRES_DB}")

## 3. Feature Selection Utilities

In [ ]:
# ID / metadata columns to always remove regardless of table
ID_COLS = {
    'variant_id', 'sv_id', 'gene_symbol', 'gene_name', 'gene_full_name',
    'official_gene_symbol', 'official_symbol', 'validated_gene_symbol',
    'pharmgkb_name', 'chromosome', 'position', 'study_id', 'variant_name',
    'variant_key', 'assembly', 'reference_allele', 'alternate_allele',
    'protein_name', 'refseq_protein_accession', 'uniprot_accession',
    'variant_pharmgkb_id', 'omim_id', 'mondo_id', 'orphanet_id',
    'description', 'cdna_change', 'protein_change', 'start_pos', 'end_pos'
}

# Per-table leakage columns (label-derived or target-encoding strings)
LEAKAGE_MAP = {
    'clinical_ml_features': [
        'clinical_significance_simple', 'clinvar_pathogenicity_class',
        'review_status', 'protein_impact_category',
        'x_linked_risk_modifier', 'inheritance_pathogenicity_modifier'
    ],
    'disease_ml_features': [
        'clinical_significance_simple', 'disease_enriched',
        'primary_disease', 'disease_name_enriched',
        'variant_disease_link_quality'
    ],
    'pharmacogene_ml_features': [
        'clinical_significance_simple', 'variant_type',
        'drug_response_impact', 'drug_response_frequency_context'
    ],
    'variant_impact_ml_features': [
        'clinical_significance_simple', 'clinvar_pathogenicity_class',
        'review_status', 'variant_impact_tier', 'lof_category',
        'domain_impact_severity', 'clinical_impact_priority',
        'disease_specific_priority', 'cancer_variant_priority',
        'conservation_impact_class', 'splice_impact_severity',
        'expression_impact_context', 'disease_impact_category',
        'gene_impact_burden', 'gene_lof_tolerance',
        'gene_variant_impact_priority', 'variant_name',
        'protein_change', 'cdna_change'
    ],
    'structural_variant_ml_features': [
        'sv_pathogenicity_risk', 'disease_sv_priority',
        'sv_clinical_priority', 'sv_impact_tier', 'gene_list'
    ],
    'variant_drug_response_ml_features': [
        'clinical_significance_simple', 'drug_response_priority',
        'drug_response_category', 'clinical_actionability',
        'drug_response_frequency_context', 'primary_indication_category'
    ],
    'variant_cancer_ml_features': [
        'gene_cancer_role', 'mutation_frequency_category',
        'clinvar_is_pathogenic', 'clinvar_pathogenicity',
        'somatic_vs_germline_classification',
        'expression_change_relevance'
    ],
    'variant_population_ml_features': [
        'population_priority', 'screening_recommendation',
        'frequency_category', 'frequency_tier', 'clinical_significance',
        'disease_allele_frequency', 'carrier_frequency_by_disease',
        'expression_frequency_correlation', 'gene_mutation_tolerance'
    ],
    'population_frequency_ml_features': [
        'frequency_category', 'frequency_tier', 'clinical_significance',
        'population_priority', 'screening_recommendation'
    ],
    'gene_pharmacogene_ml_features': [
        'pharmacogene_priority', 'pharmacogene_category',
        'pharmacogene_category_enhanced', 'drug_metabolism_role',
        'clinical_actionability_tier', 'variant_impact_burden',
        'drug_metabolism_tissue_expression', 'cancer_mutation_burden',
        'primary_indication_category', 'expression_breadth'
    ],
    'gene_expression_ml_features': [
        'expression_priority', 'disease_specific_expression_pattern',
        'expression_function_correlation', 'cancer_expression_relevance',
        'domain_expression_correlation'
    ],
    'gene_protein_family_ml_features': [
        'protein_family_priority', 'variant_disease_domain_correlation',
        'cancer_protein_classification', 'oncogenic_domain_alterations',
        'disease_specific_domains'
    ],
    'gene_test_availability_ml_features': [
        'test_priority', 'test_recommendation_tier',
        'disease_test_correlation', 'variant_test_coverage_level',
        'population_test_priority', 'primary_test_type'
    ],
    'transcript_expression_ml_features': [
        'expression_priority'
    ],
    'cancer_variant_ml_features': [
        'mutation_frequency_category', 'clinvar_is_pathogenic',
        'clinvar_pathogenicity', 'somatic_vs_germline_classification',
        'expression_change_relevance'
    ],
    'drug_response_ml_features': [
        'drug_response_priority', 'drug_response_category',
        'clinical_actionability', 'variant_pharmgkb_id',
        'variant_location'
    ],
    'genetic_test_ml_features': [
        'test_priority', 'test_availability_category'
    ],
    'protein_family_ml_features': [
        'protein_family_priority', 'protein_functional_category'
    ],
}


def load_sample(table_name, engine, sample_pct=10):
    """Load a sample from the gold schema table."""
    query = f"SELECT * FROM gold.{table_name} TABLESAMPLE SYSTEM ({sample_pct})"
    return pd.read_sql(query, engine)


def load_full(table_name, engine):
    """Load full table (for small tables under 100K rows)."""
    return pd.read_sql(f"SELECT * FROM gold.{table_name}", engine)


def convert_types(df, table_name, gold_schema):
    """Convert columns to correct types from gold schema."""
    table_schema = gold_schema[gold_schema['table_name'] == table_name]
    for _, row in table_schema.iterrows():
        col = row['column_name']
        dtype = row['data_type']
        if col not in df.columns:
            continue
        if dtype in ('INT', 'BIGINT'):
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
        elif dtype == 'DOUBLE':
            df[col] = pd.to_numeric(df[col], errors='coerce')
        elif dtype == 'BOOLEAN':
            df[col] = df[col].astype(str).str.lower().map({'true': True, 'false': False})
    return df


def pass1_remove_leakage(df, table_name, target_col):
    """Pass 1: remove ID columns, leakage columns, and the target."""
    leakage = set(LEAKAGE_MAP.get(table_name, []))
    remove = ID_COLS | leakage | {target_col}
    kept = [c for c in df.columns if c not in remove]
    dropped = [c for c in df.columns if c in remove and c != target_col]
    return kept, dropped


def pass2_quality_filter(df, candidates, missing_threshold=0.50):
    """Pass 2: remove high-missing and zero-variance columns."""
    removed_missing = []
    removed_zero_var = []
    kept = []
    for col in candidates:
        miss_rate = df[col].isnull().mean()
        if miss_rate > missing_threshold:
            removed_missing.append((col, round(miss_rate * 100, 1)))
            continue
        numeric_data = pd.to_numeric(df[col], errors='coerce')
        if df[col].dtype == bool or df[col].dtype == object:
            unique_vals = df[col].nunique(dropna=True)
            if unique_vals <= 1:
                removed_zero_var.append(col)
                continue
        else:
            if numeric_data.nunique(dropna=True) <= 1:
                removed_zero_var.append(col)
                continue
        kept.append(col)
    return kept, removed_missing, removed_zero_var


def pass3_correlation_filter(df, candidates, threshold=0.95):
    """Pass 3: remove one of each highly correlated pair (r > threshold)."""
    numeric_candidates = []
    for col in candidates:
        if df[col].dtype == bool or str(df[col].dtype) == 'boolean':
            numeric_candidates.append(col)
        else:
            numeric_candidates.append(col)

    numeric_df = df[candidates].apply(pd.to_numeric, errors='coerce')
    corr_matrix = numeric_df.corr().abs()

    to_remove = set()
    cols = list(corr_matrix.columns)
    for i in range(len(cols)):
        if cols[i] in to_remove:
            continue
        for j in range(i+1, len(cols)):
            if cols[j] in to_remove:
                continue
            if corr_matrix.iloc[i, j] > threshold:
                # Remove the one with lower mean absolute correlation (less informative)
                mean_i = corr_matrix[cols[i]].mean()
                mean_j = corr_matrix[cols[j]].mean()
                # Keep the one with higher mean correlation (more connected = more informative)
                # But prefer to remove if name suggests it is derived
                # Simple rule: remove j (second encountered)
                to_remove.add(cols[j])

    kept = [c for c in candidates if c not in to_remove]
    removed = list(to_remove)
    return kept, removed


def save_feature_list(uc_code, table_name, target_col, features, output_dir,
                      pass1_dropped, pass2_missing, pass2_zero, pass3_removed):
    """Save feature list CSV and print summary."""
    out_path = output_dir / f'{uc_code}_features.csv'
    feature_df = pd.DataFrame({'feature': features, 'table': table_name, 'use_case': uc_code})
    feature_df.to_csv(out_path, index=False)

    print(f"  Saved: {out_path.name}  ({len(features)} features)")
    print(f"  Pass 1 removed (leakage/ID)   : {len(pass1_dropped)}")
    print(f"  Pass 2 removed (missing >50%) : {len(pass2_missing)}")
    print(f"  Pass 2 removed (zero var)     : {len(pass2_zero)}")
    print(f"  Pass 3 removed (corr >0.95)   : {len(pass3_removed)}")
    print(f"  Final feature count           : {len(features)}")
    print()
    return {
        'use_case': uc_code,
        'table': table_name,
        'target': target_col,
        'final_feature_count': len(features),
        'pass1_leakage_removed': len(pass1_dropped),
        'pass2_missing_removed': len(pass2_missing),
        'pass2_zero_var_removed': len(pass2_zero),
        'pass3_correlation_removed': len(pass3_removed),
    }

print("Utilities defined")
print(f"Tables with leakage rules : {len(LEAKAGE_MAP)}")
print(f"ID columns always removed : {len(ID_COLS)}")

## 4. Load Gold Schema

In [ ]:
gold_schema = pd.read_sql(
    """SELECT table_name, column_name, data_type
       FROM information_schema.columns
       WHERE table_schema = 'gold'
       ORDER BY table_name, ordinal_position""",
    engine
)

# Map Postgres types back to our type names
type_map = {
    'bigint': 'INT', 'integer': 'INT', 'smallint': 'INT',
    'double precision': 'DOUBLE', 'real': 'DOUBLE', 'numeric': 'DOUBLE',
    'boolean': 'BOOLEAN',
    'character varying': 'STRING', 'text': 'STRING', 'character': 'STRING',
    'USER-DEFINED': 'STRING'
}
gold_schema['data_type'] = gold_schema['data_type'].map(type_map).fillna('STRING')

print(f"Gold schema loaded: {gold_schema['table_name'].nunique()} tables")
print(f"Total columns     : {len(gold_schema)}")

## 5. Feature Selection Summary Tracker

In [ ]:
summary_rows = []

print("Starting feature selection for all 15 use cases...")
print("=" * 65)

## 6. UC01 - Variant Pathogenicity (clinical_ml_features)

In [ ]:
print("UC01 - clinical_ml_features - target: target_is_pathogenic")
print("-" * 60)

TABLE = 'clinical_ml_features'
TARGET = 'target_is_pathogenic'

df = load_sample(TABLE, engine, 10)
df = convert_types(df, TABLE, gold_schema)

candidates_p1, dropped_p1 = pass1_remove_leakage(df, TABLE, TARGET)
candidates_p2, miss_p2, zero_p2 = pass2_quality_filter(df, candidates_p1)
final_features, removed_p3 = pass3_correlation_filter(df, candidates_p2)

row = save_feature_list('uc01_clinical', TABLE, TARGET, final_features, FEATURE_DIR,
                         dropped_p1, miss_p2, zero_p2, removed_p3)
summary_rows.append(row)

print(f"Sample features: {final_features[:5]}")

## 7. UC02 - Disease Association (disease_ml_features)

In [ ]:
print("UC02 - disease_ml_features - target: is_pathogenic")
print("-" * 60)

TABLE = 'disease_ml_features'
TARGET = 'is_pathogenic'

df = load_sample(TABLE, engine, 10)
df = convert_types(df, TABLE, gold_schema)

candidates_p1, dropped_p1 = pass1_remove_leakage(df, TABLE, TARGET)
candidates_p2, miss_p2, zero_p2 = pass2_quality_filter(df, candidates_p1)
final_features, removed_p3 = pass3_correlation_filter(df, candidates_p2)

row = save_feature_list('uc02_disease', TABLE, TARGET, final_features, FEATURE_DIR,
                         dropped_p1, miss_p2, zero_p2, removed_p3)
summary_rows.append(row)

print(f"Sample features: {final_features[:5]}")

## 8. UC03 - Pharmacogene Variant Impact (pharmacogene_ml_features)

In [ ]:
print("UC03 - pharmacogene_ml_features - target: is_pathogenic")
print("-" * 60)

TABLE = 'pharmacogene_ml_features'
TARGET = 'is_pathogenic'

df = load_sample(TABLE, engine, 10)
df = convert_types(df, TABLE, gold_schema)

candidates_p1, dropped_p1 = pass1_remove_leakage(df, TABLE, TARGET)
candidates_p2, miss_p2, zero_p2 = pass2_quality_filter(df, candidates_p1)
final_features, removed_p3 = pass3_correlation_filter(df, candidates_p2)

row = save_feature_list('uc03_pharmacogene', TABLE, TARGET, final_features, FEATURE_DIR,
                         dropped_p1, miss_p2, zero_p2, removed_p3)
summary_rows.append(row)

print(f"Sample features: {final_features[:5]}")

## 9. UC04 - Functional Impact Scoring (variant_impact_ml_features)

In [ ]:
print("UC04 - variant_impact_ml_features - target: is_high_impact")
print("-" * 60)

TABLE = 'variant_impact_ml_features'
TARGET = 'is_high_impact'

df = load_sample(TABLE, engine, 10)
df = convert_types(df, TABLE, gold_schema)

candidates_p1, dropped_p1 = pass1_remove_leakage(df, TABLE, TARGET)
candidates_p2, miss_p2, zero_p2 = pass2_quality_filter(df, candidates_p1)
final_features, removed_p3 = pass3_correlation_filter(df, candidates_p2)

row = save_feature_list('uc04_variant_impact', TABLE, TARGET, final_features, FEATURE_DIR,
                         dropped_p1, miss_p2, zero_p2, removed_p3)
summary_rows.append(row)

print(f"Sample features: {final_features[:5]}")

## 10. UC05 - Structural Variant Risk (structural_variant_ml_features)
**Note: sv_classification is multiclass. gene_list dropped and genes_overlapped used instead.**

In [ ]:
print("UC05 - structural_variant_ml_features - target: sv_classification")
print("-" * 60)

TABLE = 'structural_variant_ml_features'
TARGET = 'sv_classification'

df = load_full(TABLE, engine)
df = convert_types(df, TABLE, gold_schema)

# gene_list column: convert to count feature, then drop original
if 'gene_list' in df.columns:
    df['gene_list_count'] = df['gene_list'].fillna('').str.split('|').str.len()
    df['gene_list_count'] = df['gene_list_count'].where(df['gene_list'] != '', 0)

candidates_p1, dropped_p1 = pass1_remove_leakage(df, TABLE, TARGET)
# Add gene_list_count as it is a derived numeric feature (not leakage)
if 'gene_list_count' in df.columns and 'gene_list_count' not in candidates_p1:
    candidates_p1.append('gene_list_count')

candidates_p2, miss_p2, zero_p2 = pass2_quality_filter(df, candidates_p1)
final_features, removed_p3 = pass3_correlation_filter(df, candidates_p2)

row = save_feature_list('uc05_structural_variant', TABLE, TARGET, final_features, FEATURE_DIR,
                         dropped_p1, miss_p2, zero_p2, removed_p3)
summary_rows.append(row)

# Print class distribution for multiclass target
print("sv_classification class distribution:")
print(df[TARGET].value_counts().to_string())
print(f"\nSample features: {final_features[:5]}")

## 11. UC06 - Drug Response Variant Priority (variant_drug_response_ml_features)

In [ ]:
print("UC06 - variant_drug_response_ml_features - target: is_actionable_pharmacogene_variant")
print("-" * 60)

TABLE = 'variant_drug_response_ml_features'
TARGET = 'is_actionable_pharmacogene_variant'

df = load_sample(TABLE, engine, 10)
df = convert_types(df, TABLE, gold_schema)

candidates_p1, dropped_p1 = pass1_remove_leakage(df, TABLE, TARGET)
candidates_p2, miss_p2, zero_p2 = pass2_quality_filter(df, candidates_p1)
final_features, removed_p3 = pass3_correlation_filter(df, candidates_p2)

row = save_feature_list('uc06_drug_response_variant', TABLE, TARGET, final_features, FEATURE_DIR,
                         dropped_p1, miss_p2, zero_p2, removed_p3)
summary_rows.append(row)

print(f"Sample features: {final_features[:5]}")

## 12. UC07 - Cancer Variant Classification (variant_cancer_ml_features)

In [ ]:
print("UC07 - variant_cancer_ml_features - target: is_driver_candidate")
print("-" * 60)

TABLE = 'variant_cancer_ml_features'
TARGET = 'is_driver_candidate'

df = load_sample(TABLE, engine, 10)
df = convert_types(df, TABLE, gold_schema)

candidates_p1, dropped_p1 = pass1_remove_leakage(df, TABLE, TARGET)
candidates_p2, miss_p2, zero_p2 = pass2_quality_filter(df, candidates_p1)
final_features, removed_p3 = pass3_correlation_filter(df, candidates_p2)

row = save_feature_list('uc07_cancer_variant', TABLE, TARGET, final_features, FEATURE_DIR,
                         dropped_p1, miss_p2, zero_p2, removed_p3)
summary_rows.append(row)

print(f"Sample features: {final_features[:5]}")

## 13. UC08 - Population Carrier Screening (variant_population_ml_features)

In [ ]:
print("UC08 - variant_population_ml_features - target: is_carrier_screening_candidate")
print("-" * 60)

TABLE = 'variant_population_ml_features'
TARGET = 'is_carrier_screening_candidate'

# This table has ~46K rows, load full
df = load_full(TABLE, engine)
df = convert_types(df, TABLE, gold_schema)

candidates_p1, dropped_p1 = pass1_remove_leakage(df, TABLE, TARGET)
candidates_p2, miss_p2, zero_p2 = pass2_quality_filter(df, candidates_p1)
final_features, removed_p3 = pass3_correlation_filter(df, candidates_p2)

row = save_feature_list('uc08_carrier_screening', TABLE, TARGET, final_features, FEATURE_DIR,
                         dropped_p1, miss_p2, zero_p2, removed_p3)
summary_rows.append(row)

print(f"Sample features: {final_features[:5]}")

## 14. UC09 - Population Frequency Risk (population_frequency_ml_features)

In [ ]:
print("UC09 - population_frequency_ml_features - target: is_clinically_actionable_rare_variant")
print("-" * 60)

TABLE = 'population_frequency_ml_features'
TARGET = 'is_clinically_actionable_rare_variant'

df = load_full(TABLE, engine)
df = convert_types(df, TABLE, gold_schema)

candidates_p1, dropped_p1 = pass1_remove_leakage(df, TABLE, TARGET)
candidates_p2, miss_p2, zero_p2 = pass2_quality_filter(df, candidates_p1)
final_features, removed_p3 = pass3_correlation_filter(df, candidates_p2)

row = save_feature_list('uc09_population_frequency', TABLE, TARGET, final_features, FEATURE_DIR,
                         dropped_p1, miss_p2, zero_p2, removed_p3)
summary_rows.append(row)

print(f"Sample features: {final_features[:5]}")

## 15. UC10 - Gene Pharmacogene Priority (gene_pharmacogene_ml_features)
**Small dataset: 2,209 rows. Cross-validation required.**

In [ ]:
print("UC10 - gene_pharmacogene_ml_features - target: is_high_priority_pharmacogene")
print("-" * 60)
print("NOTE: Only ~2,209 rows. Cross-validation required for training.")

TABLE = 'gene_pharmacogene_ml_features'
TARGET = 'is_high_priority_pharmacogene'

df = load_full(TABLE, engine)
df = convert_types(df, TABLE, gold_schema)

print(f"Full table loaded: {len(df):,} rows")

candidates_p1, dropped_p1 = pass1_remove_leakage(df, TABLE, TARGET)
candidates_p2, miss_p2, zero_p2 = pass2_quality_filter(df, candidates_p1)
final_features, removed_p3 = pass3_correlation_filter(df, candidates_p2)

row = save_feature_list('uc10_gene_pharmacogene', TABLE, TARGET, final_features, FEATURE_DIR,
                         dropped_p1, miss_p2, zero_p2, removed_p3)
row['cv_required'] = True
summary_rows.append(row)

print(f"Sample features: {final_features[:5]}")

## 16. UC11 - Gene Expression Relevance (gene_expression_ml_features)

In [ ]:
print("UC11 - gene_expression_ml_features - target: is_clinically_relevant_expression")
print("-" * 60)

TABLE = 'gene_expression_ml_features'
TARGET = 'is_clinically_relevant_expression'

df = load_full(TABLE, engine)
df = convert_types(df, TABLE, gold_schema)

candidates_p1, dropped_p1 = pass1_remove_leakage(df, TABLE, TARGET)
candidates_p2, miss_p2, zero_p2 = pass2_quality_filter(df, candidates_p1)
final_features, removed_p3 = pass3_correlation_filter(df, candidates_p2)

row = save_feature_list('uc11_gene_expression', TABLE, TARGET, final_features, FEATURE_DIR,
                         dropped_p1, miss_p2, zero_p2, removed_p3)
summary_rows.append(row)

print(f"Sample features: {final_features[:5]}")

## 17. UC12 - Protein Family Druggability (gene_protein_family_ml_features)

In [ ]:
print("UC12 - gene_protein_family_ml_features - target: is_high_value_protein_family")
print("-" * 60)

TABLE = 'gene_protein_family_ml_features'
TARGET = 'is_high_value_protein_family'

df = load_full(TABLE, engine)
df = convert_types(df, TABLE, gold_schema)

candidates_p1, dropped_p1 = pass1_remove_leakage(df, TABLE, TARGET)
candidates_p2, miss_p2, zero_p2 = pass2_quality_filter(df, candidates_p1)
final_features, removed_p3 = pass3_correlation_filter(df, candidates_p2)

row = save_feature_list('uc12_protein_family', TABLE, TARGET, final_features, FEATURE_DIR,
                         dropped_p1, miss_p2, zero_p2, removed_p3)
summary_rows.append(row)

print(f"Sample features: {final_features[:5]}")

## 18. UC13 - Genetic Test Availability Priority (gene_test_availability_ml_features)

In [ ]:
print("UC13 - gene_test_availability_ml_features - target: is_high_priority_test_gene")
print("-" * 60)

TABLE = 'gene_test_availability_ml_features'
TARGET = 'is_high_priority_test_gene'

df = load_full(TABLE, engine)
df = convert_types(df, TABLE, gold_schema)

candidates_p1, dropped_p1 = pass1_remove_leakage(df, TABLE, TARGET)
candidates_p2, miss_p2, zero_p2 = pass2_quality_filter(df, candidates_p1)
final_features, removed_p3 = pass3_correlation_filter(df, candidates_p2)

row = save_feature_list('uc13_gene_test', TABLE, TARGET, final_features, FEATURE_DIR,
                         dropped_p1, miss_p2, zero_p2, removed_p3)
summary_rows.append(row)

print(f"Sample features: {final_features[:5]}")

## 19. UC14 - Expression Pattern Simple (transcript_expression_ml_features)

In [ ]:
print("UC14 - transcript_expression_ml_features - target: is_clinically_relevant_expression")
print("-" * 60)

TABLE = 'transcript_expression_ml_features'
TARGET = 'is_clinically_relevant_expression'

df = load_full(TABLE, engine)
df = convert_types(df, TABLE, gold_schema)

candidates_p1, dropped_p1 = pass1_remove_leakage(df, TABLE, TARGET)
candidates_p2, miss_p2, zero_p2 = pass2_quality_filter(df, candidates_p1)
final_features, removed_p3 = pass3_correlation_filter(df, candidates_p2)

row = save_feature_list('uc14_expression_simple', TABLE, TARGET, final_features, FEATURE_DIR,
                         dropped_p1, miss_p2, zero_p2, removed_p3)
summary_rows.append(row)

print(f"Sample features: {final_features[:5]}")

## 20. UC15 - Cancer Variant Molecular Classification (cancer_variant_ml_features)
**Multiclass target: gene_cancer_role (oncogene/tumor_suppressor/other)**

In [ ]:
print("UC15 - cancer_variant_ml_features - target: gene_cancer_role (multiclass)")
print("-" * 60)

TABLE = 'cancer_variant_ml_features'
TARGET = 'gene_cancer_role'

df = load_sample(TABLE, engine, 10)
df = convert_types(df, TABLE, gold_schema)

candidates_p1, dropped_p1 = pass1_remove_leakage(df, TABLE, TARGET)
# For UC15 is_driver_candidate is NOT leakage (it is a feature for the multiclass model)
candidates_p2, miss_p2, zero_p2 = pass2_quality_filter(df, candidates_p1)
final_features, removed_p3 = pass3_correlation_filter(df, candidates_p2)

row = save_feature_list('uc15_cancer_molecular', TABLE, TARGET, final_features, FEATURE_DIR,
                         dropped_p1, miss_p2, zero_p2, removed_p3)
row['multiclass'] = True
summary_rows.append(row)

# Class distribution for multiclass target
print("gene_cancer_role class distribution:")
print(df[TARGET].value_counts().to_string())
print(f"\nSample features: {final_features[:5]}")

## 21. Summary Report

In [ ]:
summary_df = pd.DataFrame(summary_rows)

# Fill missing columns
for col in ['cv_required', 'multiclass']:
    if col not in summary_df.columns:
        summary_df[col] = False
summary_df['cv_required']  = summary_df['cv_required'].fillna(False)
summary_df['multiclass']   = summary_df['multiclass'].fillna(False)

summary_path = FEATURE_DIR / 'feature_selection_summary.csv'
summary_df.to_csv(summary_path, index=False)
print(f"Saved: {summary_path}")
print()
print("=" * 70)
print("FEATURE SELECTION SUMMARY - ALL 15 USE CASES")
print("=" * 70)
print(summary_df[['use_case','table','target','final_feature_count',
                   'pass1_leakage_removed','pass2_missing_removed',
                   'pass3_correlation_removed']].to_string(index=False))
print()
print("Feature list files saved:")
for f in sorted(FEATURE_DIR.glob('uc*.csv')):
    feat_df = pd.read_csv(f)
    print(f"  {f.name:<50} {len(feat_df):>3} features")

In [ ]:
# Visualization: feature count per use case
fig, ax = plt.subplots(figsize=(14, 7))

use_cases = summary_df['use_case'].tolist()
feat_counts = summary_df['final_feature_count'].tolist()
colors = plt.cm.Set2(np.linspace(0, 1, len(use_cases)))

bars = ax.barh(use_cases, feat_counts, color=colors, alpha=0.85, edgecolor='black')
for bar, val in zip(bars, feat_counts):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2.,
            str(val), va='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Number of Approved Features', fontsize=11, fontweight='bold')
ax.set_title('Approved Feature Count per Use Case after Feature Selection', fontsize=12, fontweight='bold')
ax.set_xlim(0, max(feat_counts) * 1.15)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(FEATURE_DIR / 'feature_selection_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: feature_selection_chart.png")

In [ ]:
print()
print("=" * 65)
print("FEATURE SELECTION COMPLETE")
print("=" * 65)
print()
print(f"Feature lists saved to : {FEATURE_DIR}")
print(f"Total use cases        : {len(summary_rows)}")
print(f"Total files created    : {len(list(FEATURE_DIR.glob('uc*.csv')))} CSVs + 1 summary + 1 chart")
print()
print("Cross-validation required:")
cv_cases = summary_df[summary_df['cv_required'] == True]
if len(cv_cases) > 0:
    for _, row in cv_cases.iterrows():
        print(f"  {row['use_case']} ({row['table']}) - {row['final_feature_count']} features")
else:
    print("  None flagged")
print()
print("Multiclass use cases:")
mc_cases = summary_df[summary_df['multiclass'] == True]
if len(mc_cases) > 0:
    for _, row in mc_cases.iterrows():
        print(f"  {row['use_case']} - target: {row['target']}")
else:
    print("  None flagged")
print()
print("Next: 10_train_variant_pathogenicity_models.ipynb (UC01-UC04)")